In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


#import libraries
import pandas as pd

#read both matchups files
matchups_2024 = pd.read_json("/lakehouse/default/Files/matchups_2024.json")
matchups_2025 = pd.read_json("/lakehouse/default/Files/matchups_2025.json")

#add the season column 
matchups_2024["season"] = 2024
matchups_2025["season"] = 2025

#combine both 24 and 25
matchups = pd.concat([matchups_2024, matchups_2025], ignore_index=True)
matchups.head()


StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 3, Finished, Available, Finished, False)

,points,players,roster_id,custom_points,matchup_id,starters,starters_points,players_points,week,season
0,132.44,"[10236, 11557, 11559, 11579, 11588, 11606, 116...",1,NaN,6.0,"[7523, 9221, 6806, 6786, 11631, 8126, 10236, 7...","[11.28, 17.4, 22.9, 13.6, 14.7, 11.8, 2.6, 13....","{'10236': 2.6, '11557': 0.0, '11559': 0.0, '11...",1,2024
1,142.94,"[11306, 11377, 11439, 11563, 11568, 11576, 115...",2,NaN,3.0,"[4892, 9509, 7543, 11628, 4037, 11632, 11596, ...","[29.66, 16.1, 11.9, 1.4, 22.3, 11.6, 0.0, 11.5...","{'11306': 0.0, '11377': 0.0, '11439': 5.8, '11...",1,2024
2,174.26,"[11562, 11574, 11577, 11582, 11586, 11617, 116...",3,NaN,1.0,"[9229, 4866, 8138, 7547, 6819, 2216, 8110, 116...","[27.08, 33.2, 13.3, 4.3, 7.1, 23.1, 6.0, 20.8,...","{'11562': 0.0, '11574': 0.0, '11577': 0.0, '11...",1,2024
3,176.04,"[10857, 10870, 10871, 11566, 11571, 11581, 115...",4,NaN,1.0,"[4984, 9226, 7611, 5859, 7090, 7670, 4217, 813...","[31.18, 23.0, 21.6, 22.9, 2.5, 3.5, 10.0, 4.6,...","{'10857': 0.0, '10870': 0.0, '10871': 0.0, '11...",1,2024
4,120.16,"[10444, 10863, 10866, 11567, 11627, 11716, 123...",5,NaN,5.0,"[4881, 8155, 8151, 8144, 6803, 8137, 5001, 975...","[25.12, 18.3, 18.9, 3.1, 4.8, 13.5, 6.1, 6.5, ...","{'10444': 1.3, '10863': 0.0, '10866': 0.0, '11...",1,2024


In [2]:
# create roster key
matchups["roster_key"] = (
    matchups["season"].astype(str)
    + "_"
    + matchups["roster_id"].astype(str)
)

matchups.head()

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 4, Finished, Available, Finished, False)

,points,players,roster_id,custom_points,matchup_id,starters,starters_points,players_points,week,season,roster_key
0,132.44,"[10236, 11557, 11559, 11579, 11588, 11606, 116...",1,NaN,6.0,"[7523, 9221, 6806, 6786, 11631, 8126, 10236, 7...","[11.28, 17.4, 22.9, 13.6, 14.7, 11.8, 2.6, 13....","{'10236': 2.6, '11557': 0.0, '11559': 0.0, '11...",1,2024,2024_1
1,142.94,"[11306, 11377, 11439, 11563, 11568, 11576, 115...",2,NaN,3.0,"[4892, 9509, 7543, 11628, 4037, 11632, 11596, ...","[29.66, 16.1, 11.9, 1.4, 22.3, 11.6, 0.0, 11.5...","{'11306': 0.0, '11377': 0.0, '11439': 5.8, '11...",1,2024,2024_2
2,174.26,"[11562, 11574, 11577, 11582, 11586, 11617, 116...",3,NaN,1.0,"[9229, 4866, 8138, 7547, 6819, 2216, 8110, 116...","[27.08, 33.2, 13.3, 4.3, 7.1, 23.1, 6.0, 20.8,...","{'11562': 0.0, '11574': 0.0, '11577': 0.0, '11...",1,2024,2024_3
3,176.04,"[10857, 10870, 10871, 11566, 11571, 11581, 115...",4,NaN,1.0,"[4984, 9226, 7611, 5859, 7090, 7670, 4217, 813...","[31.18, 23.0, 21.6, 22.9, 2.5, 3.5, 10.0, 4.6,...","{'10857': 0.0, '10870': 0.0, '10871': 0.0, '11...",1,2024,2024_4
4,120.16,"[10444, 10863, 10866, 11567, 11627, 11716, 123...",5,NaN,5.0,"[4881, 8155, 8151, 8144, 6803, 8137, 5001, 975...","[25.12, 18.3, 18.9, 3.1, 4.8, 13.5, 6.1, 6.5, ...","{'10444': 1.3, '10863': 0.0, '10866': 0.0, '11...",1,2024,2024_5


In [3]:
# Create empty columns
matchups["result"] = None
matchups["point_difference"] = None
matchups["opponent_roster_id"] = None
matchups["opponent_points"] = None

# Process each matchup (each matchup has two teams)
for (season, week, matchup_id), group in matchups.groupby(["season", "week", "matchup_id"]):

    # Skip incomplete matchups
    if len(group) != 2:
        continue

    idx1, idx2 = group.index

    team1 = group.loc[idx1]
    team2 = group.loc[idx2]

    # Opponent info
    matchups.loc[idx1, "opponent_roster_id"] = team2["roster_id"]
    matchups.loc[idx2, "opponent_roster_id"] = team1["roster_id"]

    matchups.loc[idx1, "opponent_points"] = team2["points"]
    matchups.loc[idx2, "opponent_points"] = team1["points"]

    # Point differential
    diff = team1["points"] - team2["points"]

    matchups.loc[idx1, "point_difference"] = diff
    matchups.loc[idx2, "point_difference"] = -diff

    # Win/Loss/Tie
    if diff > 0:
        matchups.loc[idx1, "result"] = "Win"
        matchups.loc[idx2, "result"] = "Loss"
    elif diff < 0:
        matchups.loc[idx1, "result"] = "Loss"
        matchups.loc[idx2, "result"] = "Win"
    else:
        matchups.loc[idx1, "result"] = "Tie"
        matchups.loc[idx2, "result"] = "Tie"

matchups.head()

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 5, Finished, Available, Finished, False)

,points,players,roster_id,custom_points,matchup_id,starters,starters_points,players_points,week,season,roster_key,result,point_difference,opponent_roster_id,opponent_points
0,132.44,"[10236, 11557, 11559, 11579, 11588, 11606, 116...",1,NaN,6.0,"[7523, 9221, 6806, 6786, 11631, 8126, 10236, 7...","[11.28, 17.4, 22.9, 13.6, 14.7, 11.8, 2.6, 13....","{'10236': 2.6, '11557': 0.0, '11559': 0.0, '11...",1,2024,2024_1,Win,54.08,8,78.36
1,142.94,"[11306, 11377, 11439, 11563, 11568, 11576, 115...",2,NaN,3.0,"[4892, 9509, 7543, 11628, 4037, 11632, 11596, ...","[29.66, 16.1, 11.9, 1.4, 22.3, 11.6, 0.0, 11.5...","{'11306': 0.0, '11377': 0.0, '11439': 5.8, '11...",1,2024,2024_2,Win,35.62,6,107.32
2,174.26,"[11562, 11574, 11577, 11582, 11586, 11617, 116...",3,NaN,1.0,"[9229, 4866, 8138, 7547, 6819, 2216, 8110, 116...","[27.08, 33.2, 13.3, 4.3, 7.1, 23.1, 6.0, 20.8,...","{'11562': 0.0, '11574': 0.0, '11577': 0.0, '11...",1,2024,2024_3,Loss,-1.78,4,176.04
3,176.04,"[10857, 10870, 10871, 11566, 11571, 11581, 115...",4,NaN,1.0,"[4984, 9226, 7611, 5859, 7090, 7670, 4217, 813...","[31.18, 23.0, 21.6, 22.9, 2.5, 3.5, 10.0, 4.6,...","{'10857': 0.0, '10870': 0.0, '10871': 0.0, '11...",1,2024,2024_4,Win,1.78,3,174.26
4,120.16,"[10444, 10863, 10866, 11567, 11627, 11716, 123...",5,NaN,5.0,"[4881, 8155, 8151, 8144, 6803, 8137, 5001, 975...","[25.12, 18.3, 18.9, 3.1, 4.8, 13.5, 6.1, 6.5, ...","{'10444': 1.3, '10863': 0.0, '10866': 0.0, '11...",1,2024,2024_5,Loss,-29.48,12,149.64


In [4]:
# Check for matchup records where fantasy points are missing.
# Used as a data quality validation step before creating the fact table.
matchups[matchups["points"].isna()]

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 6, Finished, Available, Finished, False)

,points,players,roster_id,custom_points,matchup_id,starters,starters_points,players_points,week,season,roster_key,result,point_difference,opponent_roster_id,opponent_points


In [5]:
# Create a unique season-week identifier for time-based analysis.
# Combining season and week prevents duplicate week numbers across seasons
# and supports relationships with the calendar dimension.
matchups["season_week_key"] = (
    matchups["season"].astype(str)
    + "_"
    + matchups["week"].astype(str).str.zfill(2)
)

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 7, Finished, Available, Finished, False)

In [6]:
#save as a delta table!!!!
from delta.tables import DeltaTable

spark_df = spark.createDataFrame(matchups)
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Fact_Matchups")

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 8, Finished, Available, Finished, False)

In [7]:
### checking dataframe
matchups.dtypes

StatementMeta(, c170e45c-c955-468d-b513-8a9522d618ae, 9, Finished, Available, Finished, False)

points                float64
players                object
roster_id               int64
custom_points         float64
matchup_id            float64
starters               object
starters_points        object
players_points         object
week                    int64
season                  int64
roster_key             object
result                 object
point_difference       object
opponent_roster_id     object
opponent_points        object
season_week_key        object
dtype: object